In [2]:
# ====================================================
# NeurIPS 2025 - Open Polymer Prediction
# Phase1 Refactored: Fast & Clean
# ====================================================

import warnings
warnings.filterwarnings("ignore")   # Suppress Python warnings

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')      # Suppress RDKit warnings

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error


In [3]:
# Load competition data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

TARGETS = ["Tg","FFV","Tc","Density","Rg"]

print("Train:", train.shape)
print("Test :", test.shape)


Train: (7973, 7)
Test : (3, 2)


In [4]:
def competition_wmae(y_true_dict, y_pred_dict):
    maes, weights, total_weight = {}, {}, 0.0
    for target in TARGETS:
        y_true_full = np.asarray(y_true_dict[target])
        y_pred_full = np.asarray(y_pred_dict[target])

        mask = (~np.isnan(y_true_full)) & np.isfinite(y_pred_full)
        if mask.sum() == 0: continue

        y_true, y_pred = y_true_full[mask], y_pred_full[mask]
        mae = mean_absolute_error(y_true, y_pred)

        value_range = np.nanmax(y_true) - np.nanmin(y_true)
        if value_range == 0: value_range = 1.0

        weight = (1.0 / np.sqrt(mask.sum())) / value_range
        maes[target] = mae
        weights[target] = weight
        total_weight += weight

    for t in weights:
        weights[t] = weights[t] / total_weight if total_weight > 0 else 0.0

    wmae = sum(maes[t] * weights[t] for t in maes)
    return wmae, maes


In [5]:
from rdkit import Chem
from concurrent.futures import ProcessPoolExecutor

def smiles_to_mol(s):
    if not isinstance(s, str): return None
    s2 = s.replace("*","C")  # polymer wildcard handling
    try:
        return Chem.MolFromSmiles(s2)
    except:
        return None

def smiles_to_mols_parallel(smiles_list, max_workers=8):
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        mols = list(executor.map(smiles_to_mol, smiles_list))
    return mols

print("Converting SMILES to Mol objects...")
train_mols = smiles_to_mols_parallel(train["SMILES"].tolist(), max_workers=8)
test_mols  = smiles_to_mols_parallel(test["SMILES"].tolist(), max_workers=8)


Converting SMILES to Mol objects...


In [6]:
from mordred import Calculator, descriptors

calc = Calculator(descriptors, ignore_3D=True)

print("Computing Mordred descriptors...")
train_desc = calc.pandas(train_mols, nproc=8).apply(pd.to_numeric, errors="coerce")
test_desc  = calc.pandas(test_mols, nproc=8).apply(pd.to_numeric, errors="coerce")

# Drop all-NaN and constant columns
train_desc = train_desc.dropna(axis=1, how="all")
nunique = train_desc.nunique(dropna=True)
train_desc = train_desc.loc[:, nunique > 1]

# Align test
test_desc = test_desc.reindex(columns=train_desc.columns)

# Median imputation
med = train_desc.median(numeric_only=True)
train_desc = train_desc.fillna(med)
test_desc  = test_desc.fillna(med)

print("✅ Mordred features:", train_desc.shape, test_desc.shape)


Computing Mordred descriptors...


100%|██████████| 3/3 [00:00<00:00, 13.67it/s]


✅ Mordred features: (7973, 1412) (3, 1412)


In [7]:
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from joblib import Parallel, delayed

def morgan_count_fp(mol, radius=2, nBits=1024):
    if mol is None: return np.zeros(nBits, dtype=float)
    fp = AllChem.GetHashedMorganFingerprint(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=int)
    for idx, count in fp.GetNonzeroElements().items():
        arr[idx % nBits] = count
    return arr.astype(float)

def rdkit_basic(mol):
    if mol is None: return np.array([np.nan]*8, dtype=float)
    try:
        return np.array([
            Descriptors.MolWt(mol),
            Descriptors.TPSA(mol),
            Descriptors.MolLogP(mol),
            rdMolDescriptors.CalcNumRings(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
            rdMolDescriptors.CalcNumAliphaticRings(mol),
            rdMolDescriptors.CalcNumHBD(mol),
            rdMolDescriptors.CalcNumHBA(mol),
        ], dtype=float)
    except:
        return np.array([np.nan]*8, dtype=float)

def build_extra_features(mols, n_jobs=8):
    X_fp = Parallel(n_jobs=n_jobs)(delayed(morgan_count_fp)(m) for m in mols)
    X_rd = Parallel(n_jobs=n_jobs)(delayed(rdkit_basic)(m) for m in mols)
    X_fp, X_rd = np.vstack(X_fp), np.vstack(X_rd)

    # median impute
    col_med = np.nanmedian(X_rd, axis=0)
    inds = np.where(np.isnan(X_rd))
    X_rd[inds] = np.take(col_med, inds[1])

    return np.hstack([X_fp, X_rd])

print("Building extra features...")
X_train_extra = build_extra_features(train_mols, n_jobs=8)
X_test_extra  = build_extra_features(test_mols, n_jobs=8)

# Combine Mordred + extra features
X_train_all = np.hstack([train_desc.values, X_train_extra])
X_test_all  = np.hstack([test_desc.values,  X_test_extra])

print("✅ Combined features:", X_train_all.shape, X_test_all.shape)


Building extra features...


[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerator
[16:15:32] DEPRECATION WARNING: please use MorganGenerat

✅ Combined features: (7973, 2444) (3, 2444)


In [14]:
from lightgbm import LGBMRegressor
import lightgbm as lgb   # <-- needed for callbacks
import gc
def train_lgbm_per_target(X, y_dict, X_test, n_splits=5, seed=42):
    preds_test = {}
    oof_preds = {t: np.full(len(y_dict[t]), np.nan) for t in TARGETS}

    for target in TARGETS:
        print(f"\n🔹 Training target: {target}")
        y_full = y_dict[target].values
        mask = ~np.isnan(y_full)
        X_masked, y = X[mask], y_full[mask]

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        preds_test_target = []

        for fold, (trn_idx, val_idx) in enumerate(kf.split(X_masked, y)):
            print(f"  Fold {fold+1}/{n_splits}...")

            X_tr, X_val = X_masked[trn_idx], X_masked[val_idx]
            y_tr, y_val = y[trn_idx], y[val_idx]

            model = LGBMRegressor(
                objective="regression",
                metric="mae",
                learning_rate=0.05,
                num_leaves=64,
                feature_fraction=0.8,
                bagging_fraction=0.8,
                bagging_freq=5,
                n_estimators=5000,
                random_state=seed,
                n_jobs=-1
            )

            # ✅ use callbacks for early stopping
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                eval_metric="mae",
                callbacks=[lgb.early_stopping(200, verbose=False)]
            )

            # 🔑 FIX: correctly insert into OOF with true indices
            true_val_idx = np.where(mask)[0][val_idx]
            oof_preds[target][true_val_idx] = model.predict(
                X_val, num_iteration=model.best_iteration_
            )

            preds_test_target.append(
                model.predict(X_test, num_iteration=model.best_iteration_)
            )

            del X_tr, X_val, y_tr, y_val
            gc.collect()

        preds_test[target] = np.mean(preds_test_target, axis=0)

    return oof_preds, preds_test



In [15]:
y_dict = {t: train[t] for t in TARGETS}
oof_preds, test_preds = train_lgbm_per_target(X_train_all, y_dict, X_test_all)

wmae, maes = competition_wmae(y_dict, oof_preds)
print("\n✅ OOF wMAE:", round(wmae, 5))
print("Per-target MAEs:")
for t, v in maes.items():
    print(f"  {t}: {v:.4f}")



🔹 Training target: Tg
  Fold 1/5...
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012478 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 142491
[LightGBM] [Info] Number of data points in the train set: 408, numb